In [8]:
import pandas as pd

df=pd.read_csv("online_retail_II.csv")

#Checking the Data
print("Data Review")
print(df.head())

print(len(df))

Data Review
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS      12.0   
1  489434    79323P                   PINK CHERRY LIGHTS      12.0   
2  489434    79323W                  WHITE CHERRY LIGHTS      12.0   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE       48.0   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX      24.0   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
35196


In [9]:

# Data Cleaning

#1 Checking Null
print(df.isnull().sum())

#2 Dropping Null Rows
df_clean=df.dropna(subset=["Customer ID"]).copy()

#3 Filtering Out Cancelled orders
df_clean=df_clean[(df_clean["Quantity"]>0) & (df_clean["Price"]>0)]

#4 Creating new columns for calculations
df_clean["Total_Spend"]=df_clean["Quantity"] * df_clean["Price"]

#5 Converting dates to proper format
df_clean["InvoiceDate"]=pd.to_datetime(df["InvoiceDate"],errors="coerce")

print(f"Cleaned Data Set has {len(df_clean)} rows")

Invoice           0
StockCode         1
Description      75
Quantity          1
InvoiceDate       1
Price             1
Customer ID    9189
Country           1
dtype: int64
Cleaned Data Set has 25222 rows


In [10]:
#RFM Metrics
snapshot_date=df_clean["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm=(df_clean.groupby("Customer ID").agg(
    {
        "InvoiceDate":lambda x:(snapshot_date - x.max()).days, #days since last order
        "Invoice":"nunique",
        "Total_Spend":"sum"
    }
).reset_index()
)
rfm.columns=("Customer ID","Recency_Days","Frequency","Monetary_Value")
rfm.head()

# Save to a new CSV file to use for your dashboard
rfm.to_csv("rfm_customer_segments.csv", index=False)
df_clean.to_csv("clean_retail_transactions.csv", index=False)
print("Files saved! You can download them from the left sidebar.")

rfm.head()


Files saved! You can download them from the left sidebar.


,Customer ID,Recency_Days,Frequency,Monetary_Value
0,12346.0,3,3,90.00
1,12358.0,9,1,1429.83
2,12359.0,11,1,760.69
3,12362.0,16,1,130.00
4,12417.0,7,2,317.60


In [11]:
#Setting up the labels
rfm["R_Score"]=pd.qcut(rfm["Recency_Days"],q=4,labels=[4,3,2,1])
rfm["F_Score"]=pd.qcut(rfm["Frequency"].rank(method="first"),q=4,labels=[1,2,3,4])
rfm["M_Score"]=pd.qcut(rfm["Monetary_Value"],q=4,labels=[1,2,3,4])

#Grouping
rfm["RFM_Group"]=(rfm["R_Score"].astype(str)+
              rfm["F_Score"].astype(str)+
              rfm["M_Score"].astype(str))

rfm["RFM_Total_Score"]=(rfm['R_Score'].astype(int)+
                        rfm["F_Score"].astype(int)+
                        rfm["M_Score"].astype(int))
display(rfm)

,Customer ID,Recency_Days,Frequency,Monetary_Value,R_Score,F_Score,M_Score,RFM_Group,RFM_Total_Score
0,12346.0,3,3,90.00,4,4,1,441,9
1,12358.0,9,1,1429.83,2,1,4,214,7
2,12359.0,11,1,760.69,2,1,4,214,7
3,12362.0,16,1,130.00,1,1,1,111,3
4,12417.0,7,2,317.60,3,3,2,332,8
...,...,...,...,...,...,...,...,...,...
847,18268.0,6,1,375.18,3,3,3,333,9
848,18271.0,9,1,192.55,2,3,1,231,6
849,18275.0,8,1,372.64,3,3,3,333,9
850,18276.0,6,1,218.05,3,3,2,332,8


In [12]:
# Function to assign segment names based on scores
def assign_segment(df):
    if df["RFM_Total_Score"] >= 10:
        return "Champions"
    elif (df["RFM_Total_Score"] >= 8) and (df["RFM_Total_Score"] < 10):
        return "Loyal Customers"
    elif (df["RFM_Total_Score"] >= 6) and (df["RFM_Total_Score"] < 8):
        return "Potential Loyalists"
    elif (df["RFM_Total_Score"] >= 4) and (df["RFM_Total_Score"] < 6):
        return "At Risk"
    else:
        return "Lost Customers"


# Apply the function to create a new 'Segment' column
rfm["Segment"] = rfm.apply(assign_segment, axis=1)

# Check the distribution of customer segments
print(rfm["Segment"].value_counts())

Segment
Potential Loyalists    262
Loyal Customers        202
Champions              200
At Risk                169
Lost Customers          19
Name: count, dtype: int64


In [13]:
rfm.to_csv("rfm_customer_segments.csv", index=False)
print("Final segmented dataset saved successfully!")

Final segmented dataset saved successfully!


In [14]:
import duckdb

con=duckdb.connect()
con.register("rfm_table",rfm)
con.register("transaction",df_clean)

In [15]:
query_1="""select Segment,count("Customer ID") as Total_Customers,
round(sum(Monetary_Value),2) as Total_Revenue,
round(avg(Monetary_Value),2) as Avg_Spend_Per_Customer
from rfm_table
group by segment
order by Total_Revenue desc;"""

con.execute(query_1).df()

,Segment,Total_Customers,Total_Revenue,Avg_Spend_Per_Customer
0,Champions,200,325445.47,1627.23
1,Loyal Customers,202,100433.17,497.19
2,Potential Loyalists,262,91918.27,350.83
3,At Risk,169,35266.71,208.68
4,Lost Customers,19,2271.29,119.54


In [21]:
query_2 = """
SELECT
    Description,
    SUM(Quantity) AS Total_Units_Sold,
    ROUND(SUM(Total_Spend), 2) AS Total_Revenue
FROM transaction
WHERE Description IS NOT NULL
GROUP BY Description
ORDER BY Total_Revenue DESC
LIMIT 10;
"""

con.execute(query_2).df()

,Description,Total_Units_Sold,Total_Revenue
0,WHITE HANGING HEART T-LIGHT HOLDER,5489.0,14477.81
1,EDWARDIAN PARASOL BLACK,2143.0,8382.85
2,PAPER CHAIN KIT 50'S CHRISTMAS,2858.0,7372.40
3,ASSORTED COLOUR BIRD ORNAMENT,4354.0,6820.66
4,JUMBO BAG RED WHITE SPOTTY,3454.0,5982.30
5,EDWARDIAN PARASOL RED,1653.0,5977.35
6,RED/WHITE DOTS RUFFLED UMBRELLA,1576.0,5852.80
7,EDWARDIAN PARASOL NATURAL,1623.0,5846.85
8,SCOTTIE DOG HOT WATER BOTTLE,1129.0,5193.75
9,BLACK/BLUE DOTS RUFFLED UMBRELLA,1455.0,5120.85


In [20]:
con.execute("SELECT * FROM transaction").df()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total_Spend
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12.0,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.0,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.00
...,...,...,...,...,...,...,...,...,...
25217,492340,15056BL,EDWARDIAN PARASOL BLACK,1.0,2009-12-16 12:29:00,5.95,17954.0,United Kingdom,5.95
25218,492340,85014B,RED/WHITE DOTS RUFFLED UMBRELLA,1.0,2009-12-16 12:29:00,5.95,17954.0,United Kingdom,5.95
25219,492340,85014A,BLACK/BLUE DOTS RUFFLED UMBRELLA,1.0,2009-12-16 12:29:00,5.95,17954.0,United Kingdom,5.95
25220,492340,85014D,PINK/BROWN DOTS RUFFLED UMBRELLA,1.0,2009-12-16 12:29:00,5.95,17954.0,United Kingdom,5.95


In [23]:
query_3 = """
SELECT
    STRFTIME(InvoiceDate, '%Y-%m') AS Year_Month,
    COUNT(DISTINCT Invoice) AS Total_Orders,
    ROUND(SUM(Total_Spend), 2) AS Monthly_Revenue
FROM transaction
GROUP BY Year_Month
ORDER BY Year_Month ASC;
"""

con.execute(query_3).df()

,Year_Month,Total_Orders,Monthly_Revenue
0,2009-12,1221,555334.91


In [27]:
from google.colab import files

# This will trigger an automatic download to your computer's Downloads folder
files.download("rfm_customer_segments.csv")
files.download("clean_retail_transactions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>